##### imports

In [136]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import shutil
import random

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score,precision_score,f1_score,recall_score,roc_auc_score

To-Do: attempting to control overfitting from baseline model

### Data Loading

In [5]:
og_df = pd.read_csv("../../data/oc_risk50k_single_dataset.csv")
df = og_df.copy()

print(df.shape,"\n\n")
df.head(5)

(50000, 52) 




,Patient_ID,Age,Menopausal_Status,Age_At_Menarche,Age_At_Menopause,Family_History,BRCA1_Status,BRCA2_Status,Lynch_Syndrome_Status,BMI,...,Charlson_Comorbidity_Index,ECOG_Performance_Status,Creatinine,Liver_Enzyme_Level,Treatment_Status,Time_To_Progression,Follow_Up_Status,Partition,Ovarian_Cancer_Status,Risk_Category
0,PAT_00001,53.8,1,11.0,48.6,0,0,0,0,34.5,...,1,2,0.98,30.6,0,-1.0,2,Train,0,0
1,PAT_00002,66.8,1,13.2,50.4,1,0,0,0,18.5,...,1,1,1.01,33.7,3,25.1,0,Train,1,1
2,PAT_00003,51.9,1,13.2,50.9,1,0,0,0,23.2,...,0,2,0.73,17.8,2,5.5,0,Train,1,2
3,PAT_00004,67.6,1,12.3,51.1,0,0,0,0,24.7,...,2,1,1.01,25.3,2,70.0,0,Test,1,2
4,PAT_00005,80.5,1,14.9,52.1,1,0,0,0,22.0,...,0,0,0.85,30.1,0,-1.0,1,Test,0,0


In [38]:
# SORT COLUMNS 

# leakage columns
cols_removed = ["Patient_ID","Risk_Category",
                "Follow_Up_Status",
                "Treatment_Status",
                "Time_To_Progression"]
try:
    df=df.drop(labels=cols_removed,axis=1)
except KeyError:
    pass

# In order of priority:
demographic_cols = ['Age', 'Menopausal_Status', 
                    'Age_At_Menarche', 
                    'Age_At_Menopause', # potentially redundant
                    "BMI", "Parity",
                    "Oral_Contraceptive_History",
                    "Oral_Contraceptive_Duration_Years",
                    "Endometriosis_History",
                    "Hormone_Therapy_History",
                    "Tubal_Ligation_History"]

medicalhist_cols = ["Family_History","BRCA1_Status","BRCA2_Status","Lynch_Syndrome_Status"]

symptom_cols = ["Abdominal_Bloating",
                "Pelvic_Pain", 
                "Early_Satiety", 
                "Urinary_Urgency", 
                "Unexplained_Weight_Loss"]

blood_cols = ["LDH", "Hemoglobin", 
              "Platelet_Count", 
              "White_Blood_Cell_Count", 
              "Albumin", 
              "C_Reactive_Protein"]

# technically imaging, but don't describe a tumor
misc_cols = ["Lymph_Node_Enlargement", "Ascites_Status"] 

biomarker_cols = ["CA125","HE4","CEA","CA19_9","AFP"]

imaging_cols = ["Tumor_Size",
                "Laterality",
                "Tumor_Morphology",
                "Solid_Component", 
                "Bilateral_Mass", 
                "Papillary_Projection", 
                "Doppler_Vascularity",
                "Imaging_Risk_Score"]

target = "Ovarian_Cancer_Status"

## Demographic Features

In [104]:
df[demographic_cols+["Ovarian_Cancer_Status"]].corr()

,Age,Menopausal_Status,Age_At_Menarche,Age_At_Menopause,BMI,Parity,Oral_Contraceptive_History,Oral_Contraceptive_Duration_Years,Endometriosis_History,Hormone_Therapy_History,Tubal_Ligation_History,Ovarian_Cancer_Status
Age,1.000000,0.744188,0.007650,0.738463,-0.000901,0.002684,-0.064302,-0.048359,0.051596,0.150760,-0.050527,0.260211
Menopausal_Status,0.744188,1.000000,-0.001282,0.992508,0.004194,0.005707,-0.065316,-0.046926,0.049814,0.204603,-0.044293,0.242349
Age_At_Menarche,0.007650,-0.001282,1.000000,-0.000403,-0.004649,0.003950,-0.001305,-0.002423,-0.005809,0.002879,0.005994,0.001014
Age_At_Menopause,0.738463,0.992508,-0.000403,1.000000,0.003772,0.006460,-0.064389,-0.045735,0.049298,0.203435,-0.044872,0.240458
BMI,-0.000901,0.004194,-0.004649,0.003772,1.000000,0.000406,-0.003375,-0.005646,-0.001114,-0.002398,-0.003156,0.000103
Parity,0.002684,0.005707,0.003950,0.006460,0.000406,1.000000,-0.012177,-0.012782,0.000375,0.006506,-0.000978,0.003691
Oral_Contraceptive_History,-0.064302,-0.065316,-0.001305,-0.064389,-0.003375,-0.012177,1.000000,0.734782,-0.044032,-0.016037,0.052385,-0.219927
Oral_Contraceptive_Duration_Years,-0.048359,-0.046926,-0.002423,-0.045735,-0.005646,-0.012782,0.734782,1.000000,-0.034797,-0.007581,0.035387,-0.163442
Endometriosis_History,0.051596,0.049814,-0.005809,0.049298,-0.001114,0.000375,-0.044032,-0.034797,1.000000,0.021265,-0.036988,0.183604
Hormone_Therapy_History,0.150760,0.204603,0.002879,0.203435,-0.002398,0.006506,-0.016037,-0.007581,0.021265,1.000000,-0.016009,0.052537


In [59]:
demographic_df = df[demographic_cols+[target]]
print(demographic_df.shape,"\n\n")
demographic_df.head(5)

(50000, 12) 




,Age,Menopausal_Status,Age_At_Menarche,Age_At_Menopause,BMI,Parity,Oral_Contraceptive_History,Oral_Contraceptive_Duration_Years,Endometriosis_History,Hormone_Therapy_History,Tubal_Ligation_History,Ovarian_Cancer_Status
0,53.8,1,11.0,48.6,34.5,3,1,3.1,0,0,0,0
1,66.8,1,13.2,50.4,18.5,2,0,0.0,0,0,0,1
2,51.9,1,13.2,50.9,23.2,3,0,0.0,0,0,0,1
3,67.6,1,12.3,51.1,24.7,0,0,0.0,0,0,0,1
4,80.5,1,14.9,52.1,22.0,3,1,4.1,0,0,1,0


## Modelling

In [66]:
x = demographic_df.drop(labels=target,axis=1)
y = demographic_df[target]

x.shape

(50000, 11)

In [68]:
x_train,x_test,y_train,y_test = train_test_split(x,y,train_size=0.8,random_state=1118)

print(x_train.shape,x_test.shape,y_train.shape,y_test.shape,sep="\n")

(40000, 11)
(10000, 11)
(40000,)
(10000,)


In [88]:
y.value_counts()

Ovarian_Cancer_Status
0    25000
1    25000
Name: count, dtype: int64

### LogReg

In [100]:
pipeline = make_pipeline(SimpleImputer(strategy='mean'), LogisticRegression(max_iter=1000))

pipeline.fit(x_train, y_train)

y_pred = pipeline.predict(x_test)


print("Accuracy :",round(accuracy_score(y_test,y_pred),4))
print("Precision:",round(precision_score(y_test,y_pred),4))
print("Recall   :",round(recall_score(y_test,y_pred),4))
print("F1 Score :",round(f1_score(y_test,y_pred),4))
print("ROC AUC  :",round(roc_auc_score(np.array(y_test).reshape(-1), 
              pipeline.predict_proba(x_test)[:, 1]),4)
     )

Accuracy : 0.6724
Precision: 0.6735
Recall   : 0.6662
F1 Score : 0.6698
ROC AUC  : 0.7373


In [102]:
model = pipeline.named_steps['logisticregression']
coefs = pd.Series(model.coef_[0], index=x_train.columns)

print(coefs.abs().sort_values(ascending=False))

Endometriosis_History                1.211952
Tubal_Ligation_History               1.177641
Oral_Contraceptive_History           0.889866
Menopausal_Status                    0.536912
Age                                  0.032043
Parity                               0.007911
Hormone_Therapy_History              0.004133
Oral_Contraceptive_Duration_Years    0.002906
Age_At_Menarche                      0.002758
BMI                                  0.001340
Age_At_Menopause                     0.000481
dtype: float64


### Decision Tree

In [123]:
pipeline = make_pipeline(SimpleImputer(strategy='mean'), DecisionTreeClassifier(max_depth=7,random_state=1118))

pipeline.fit(x_train, y_train)

y_pred = pipeline.predict(x_test)


print("Accuracy :",round(accuracy_score(y_test,y_pred),4))
print("Precision:",round(precision_score(y_test,y_pred),4))
print("Recall   :",round(recall_score(y_test,y_pred),4))
print("F1 Score :",round(f1_score(y_test,y_pred),4))
print("ROC AUC  :",round(roc_auc_score(np.array(y_test).reshape(-1), 
              pipeline.predict_proba(x_test)[:, 1]),4)
     )

Accuracy : 0.6684
Precision: 0.6581
Recall   : 0.6977
F1 Score : 0.6773
ROC AUC  : 0.7319


*higher recall*

### RandFor

In [138]:
pipeline = make_pipeline(SimpleImputer(strategy='mean'), RandomForestClassifier(n_estimators=25,max_depth=10,random_state=1118))

pipeline.fit(x_train, y_train)

y_pred = pipeline.predict(x_test)


print("Accuracy :",round(accuracy_score(y_test,y_pred),4))
print("Precision:",round(precision_score(y_test,y_pred),4))
print("Recall   :",round(recall_score(y_test,y_pred),4))
print("F1 Score :",round(f1_score(y_test,y_pred),4))
print("ROC AUC  :",round(roc_auc_score(np.array(y_test).reshape(-1), 
                                       pipeline.predict_proba(x_test)[:, 1]),4)
     )

Accuracy : 0.6702
Precision: 0.6711
Recall   : 0.6644
F1 Score : 0.6677
ROC AUC  : 0.737


random forest not any better than decision tree --> not justified

## Model Loop

In [162]:
# Feature sets to loop through
feature_sets = {"demog": demographic_cols,
                "medhist": medicalhist_cols,
                "symptom": symptom_cols,
                "blood": blood_cols,
                "misc": misc_cols,
                "biomarker": biomarker_cols,
                "imaging": imaging_cols}

metrics_df = pd.DataFrame(index=["Train_Acc","Test_Acc","Precision","Recall","F1_Score","ROC_AUC"])
lg_importances = {}
dt_importances = {}

for name, cols in feature_sets.items():
    x = df[cols]
    y = df[target]

    x_train,x_test,y_train,y_test = train_test_split(x,y,train_size=0.8,random_state=1118)

    models = {
        "lg": make_pipeline(SimpleImputer(strategy='mean'), StandardScaler(), LogisticRegression(max_iter=1000)),
        "dt": make_pipeline(SimpleImputer(strategy='mean'), DecisionTreeClassifier(max_depth=7,random_state=1118))
    }

    for abbr, pipeline in models.items():
        pipeline.fit(x_train, y_train)
        y_pred = pipeline.predict(x_test)

        metrics_df[f"{name}_{abbr}"] = [
            round(accuracy_score(y_train, pipeline.predict(x_train)), 4),
            round(accuracy_score(y_test,y_pred),4),
            round(precision_score(y_test,y_pred),4),
            round(recall_score(y_test,y_pred),4),
            round(f1_score(y_test,y_pred),4),
            round(roc_auc_score(np.array(y_test).reshape(-1),
                                pipeline.predict_proba(x_test)[:, 1]),4)
        ]

    # feature importances
    model = models["lg"].named_steps['logisticregression']
    lg_importances[name] = pd.Series(model.coef_[0], index=x_train.columns).abs().sort_values(ascending=False)

    model = models["dt"].named_steps['decisiontreeclassifier']
    dt_importances[name] = pd.Series(model.feature_importances_, index=x_train.columns).sort_values(ascending=False)

    print(f"===== {name.upper()} =====")
    print("\nLogReg |coef| (standardized):")
    print(lg_importances[name])
    print("\nDecision Tree importances:")
    print(dt_importances[name],"\n\n")

metrics_df

===== DEMOG =====

LogReg |coef| (standardized):
Oral_Contraceptive_History           0.436838
Tubal_Ligation_History               0.418003
Endometriosis_History                0.393607
Age                                  0.386320
Menopausal_Status                    0.242891
Age_At_Menopause                     0.020890
Parity                               0.010769
Oral_Contraceptive_Duration_Years    0.009813
BMI                                  0.007155
Age_At_Menarche                      0.003828
Hormone_Therapy_History              0.001709
dtype: float64

Decision Tree importances:
Age                                  0.424566
Oral_Contraceptive_History           0.229239
Tubal_Ligation_History               0.165209
Endometriosis_History                0.135397
BMI                                  0.013621
Age_At_Menarche                      0.011934
Age_At_Menopause                     0.007080
Oral_Contraceptive_Duration_Years    0.006569
Parity                            

,demog_lg,demog_dt,medhist_lg,medhist_dt,symptom_lg,symptom_dt,blood_lg,blood_dt,misc_lg,misc_dt,biomarker_lg,biomarker_dt,imaging_lg,imaging_dt
Train_Acc,0.6711,0.6793,0.6948,0.6948,0.8787,0.8787,0.9979,0.9955,0.7893,0.7893,0.9883,0.9935,0.9819,0.9831
Test_Acc,0.6726,0.6684,0.6917,0.6917,0.8839,0.8839,0.9979,0.9918,0.7943,0.7943,0.9877,0.9893,0.9816,0.9791
Precision,0.6737,0.6581,0.8265,0.8265,0.9048,0.9048,0.9974,0.9912,0.9541,0.9541,0.9884,0.9880,0.9843,0.9805
Recall,0.6664,0.6977,0.4834,0.4834,0.8575,0.8575,0.9984,0.9924,0.6173,0.6173,0.9870,0.9906,0.9787,0.9775
F1_Score,0.6700,0.6773,0.6100,0.6100,0.8805,0.8805,0.9979,0.9918,0.7496,0.7496,0.9877,0.9893,0.9815,0.9790
ROC_AUC,0.7373,0.7319,0.7023,0.7024,0.9458,0.9457,1.0000,0.9948,0.7954,0.7954,0.9993,0.9925,0.9983,0.9953


- nothing seems to be overfitting
- recall is more meaningful than precision (50/50 dataset does not reflect real-world occurance)
- medical hist models dont seem to perform well (low recall)
- even just the 2 miscellaneous imaging columns have high performance (albeit low recall)
- blood work, biomarker, symptom, and imaging models have very high and relatively consistent metrics (expected; these, except for symptoms, will still be de-emphasized)

In [181]:
name = "noninvasive"

x = df[demographic_cols+medicalhist_cols+symptom_cols]
y = df[target]

x_train,x_test,y_train,y_test = train_test_split(x,y,train_size=0.8,random_state=1118)

models = {
    "lg": make_pipeline(SimpleImputer(strategy='mean'), StandardScaler(), LogisticRegression(max_iter=1000)),
    "dt": make_pipeline(SimpleImputer(strategy='mean'), DecisionTreeClassifier(max_depth=7,random_state=1118))
}

for abbr, pipeline in models.items():
    pipeline.fit(x_train, y_train)
    y_pred = pipeline.predict(x_test)

    metrics_df[f"{name}_{abbr}"] = [
        round(accuracy_score(y_train, pipeline.predict(x_train)), 4),
        round(accuracy_score(y_test,y_pred),4),
        round(precision_score(y_test,y_pred),4),
        round(recall_score(y_test,y_pred),4),
        round(f1_score(y_test,y_pred),4),
        round(roc_auc_score(np.array(y_test).reshape(-1),
                            pipeline.predict_proba(x_test)[:, 1]),4)
    ]

# feature importances
model = models["lg"].named_steps['logisticregression']
lg_importances[name] = pd.Series(model.coef_[0], index=x_train.columns).abs().sort_values(ascending=False)

model = models["dt"].named_steps['decisiontreeclassifier']
dt_importances[name] = pd.Series(model.feature_importances_, index=x_train.columns).sort_values(ascending=False)

print(f"===== {name.upper()} =====")
print("\nLogReg |coef| (standardized):")
print(lg_importances[name])
print("\nDecision Tree importances:")
print(dt_importances[name],"\n\n")

===== NONINVASIVE =====

LogReg |coef| (standardized):
Abdominal_Bloating                   1.377117
Pelvic_Pain                          1.275850
Early_Satiety                        1.059343
Unexplained_Weight_Loss              0.985853
BRCA1_Status                         0.763987
BRCA2_Status                         0.690630
Urinary_Urgency                      0.622049
Family_History                       0.501794
Oral_Contraceptive_History           0.461439
Tubal_Ligation_History               0.423088
Endometriosis_History                0.413911
Lynch_Syndrome_Status                0.397647
Age                                  0.397230
Menopausal_Status                    0.125873
Age_At_Menopause                     0.095498
Hormone_Therapy_History              0.028991
Oral_Contraceptive_Duration_Years    0.018493
Parity                               0.011787
Age_At_Menarche                      0.011257
BMI                                  0.008755
dtype: float64

Decision 

In [190]:
df[["BRCA1_Status","BRCA2_Status","Lynch_Syndrome_Status"]].mean() 

BRCA1_Status             0.09406
BRCA2_Status             0.07608
Lynch_Syndrome_Status    0.02696
dtype: float64

^^ relativelyy low positive occurence but highly influential; also technically require prior genetic testing which many people may not have

In [185]:
metrics_df[["noninvasive_lg",	"noninvasive_dt"]]

,noninvasive_lg,noninvasive_dt
Train_Acc,0.9106,0.9002
Test_Acc,0.9108,0.8965
Precision,0.9185,0.9016
Recall,0.9012,0.8895
F1_Score,0.9097,0.8955
ROC_AUC,0.9697,0.9512


### Removing BRCA1,2 & Lynch Symdrome Cols

In [212]:
non_invasive_cols = demographic_cols+medicalhist_cols+symptom_cols
prev_len=len(non_invasive_cols)
for x in ["BRCA1_Status","BRCA2_Status","Lynch_Syndrome_Status"]:
    non_invasive_cols.remove(x)

# also remove Age_At_Menopause --> redundant with both Age and Menopausal_Status
# and Oral_Contraceptive_Duration_Years --> redundant with Oral_Contraceptive_History and found in previous models to be insignificant
non_invasive_cols.remove("Age_At_Menopause")
non_invasive_cols.remove("Oral_Contraceptive_Duration_Years")

print("Previous num of cols:",prev_len,"\nNew num of cols:",len(non_invasive_cols))

Previous num of cols: 20 
New num of cols: 15


In [214]:
name = "noninvasive2"

x = df[non_invasive_cols]
y = df[target]

x_train,x_test,y_train,y_test = train_test_split(x,y,train_size=0.8,random_state=1118)

models = {
    "lg": make_pipeline(SimpleImputer(strategy='mean'), StandardScaler(), LogisticRegression(max_iter=1000)),
    "dt": make_pipeline(SimpleImputer(strategy='mean'), DecisionTreeClassifier(max_depth=7,random_state=1118))
}

for abbr, pipeline in models.items():
    pipeline.fit(x_train, y_train)
    y_pred = pipeline.predict(x_test)

    metrics_df[f"{name}_{abbr}"] = [
        round(accuracy_score(y_train, pipeline.predict(x_train)), 4),
        round(accuracy_score(y_test,y_pred),4),
        round(precision_score(y_test,y_pred),4),
        round(recall_score(y_test,y_pred),4),
        round(f1_score(y_test,y_pred),4),
        round(roc_auc_score(np.array(y_test).reshape(-1),
                            pipeline.predict_proba(x_test)[:, 1]),4)
    ]

# feature importances
model = models["lg"].named_steps['logisticregression']
lg_importances[name] = pd.Series(model.coef_[0], index=x_train.columns).abs().sort_values(ascending=False)

model = models["dt"].named_steps['decisiontreeclassifier']
dt_importances[name] = pd.Series(model.feature_importances_, index=x_train.columns).sort_values(ascending=False)

print(f"===== {name.upper()} =====")
print("\nLogReg |coef| (standardized):")
print(lg_importances[name])
print("\nDecision Tree importances:")
print(dt_importances[name],"\n\n")

===== NONINVASIVE2 =====

LogReg |coef| (standardized):
Abdominal_Bloating            1.382190
Pelvic_Pain                   1.271842
Early_Satiety                 1.059895
Unexplained_Weight_Loss       0.983223
Urinary_Urgency               0.612624
Family_History                0.585719
Oral_Contraceptive_History    0.433987
Tubal_Ligation_History        0.428393
Endometriosis_History         0.412783
Age                           0.383652
Menopausal_Status             0.235862
Hormone_Therapy_History       0.028767
Age_At_Menarche               0.014252
Parity                        0.010276
BMI                           0.000611
dtype: float64

Decision Tree importances:
Abdominal_Bloating            0.502830
Pelvic_Pain                   0.216189
Early_Satiety                 0.112694
Unexplained_Weight_Loss       0.073602
Urinary_Urgency               0.030011
Family_History                0.028931
Age                           0.021252
Oral_Contraceptive_History    0.004407
Tuba

In [220]:
metrics_df[["noninvasive2_lg","noninvasive2_dt"]]

,noninvasive2_lg,noninvasive2_dt
Train_Acc,0.8984,0.8933
Test_Acc,0.9005,0.8921
Precision,0.9077,0.9037
Recall,0.8911,0.8771
F1_Score,0.8993,0.8902
ROC_AUC,0.9628,0.9535


- Decision Tree:
    - very top-heavy importances (only top 3-4 are notable)
 
- Log Reg.
    - less regularized
    - better recall & ROC-AUC

- both have somewhat of the same top ~10, emphasis on symptoms (expected; symptom model performed best on its own)
- no observed overfitting

In [222]:
#metrics_df.to_csv("../results/feature_group_metrics.csv")